# [Research] Evolution of Collaborative Filtering: NeuMF to COMET

## 1. Experiment Overview

### 1.1 Paper Information
| Category | Details |
| :--- | :--- |
| **Title** | COMET: Convolutional Dimension Interaction for Collaborative Filtering |
| **Authors** | Lin et al. |
| **Published** | ACM Transactions on Intelligent Systems and Technology (TIST), 2023 |
| **Key Tech** | 2D CNN + User Interaction History |
| **Contribution** | Enhanced recommendation accuracy by learning dimensional interactions from historical behavior. |

---

### 1.2 Research Objectives
This experiment reproduces the **COMET** model to analyze the technical evolution of recommendation systems and compares it with previous benchmarks (**NeuMF, ConvNCF**).

#### Key Research Questions (RQ)
1. **Efficiency of CNN:** Is CNN-based interaction learning superior to traditional MLP?
2. **Impact of History:** Does integrating user history maps significantly boost performance?
3. **Architectural Evolution:** How does performance scale from NeuMF to COMET?

---

### 1.3 Model Evolution Roadmap

1. **NeuMF (2017)**: The foundation of Neural CF (GMF + MLP).
2. **ConvNCF (2018)**: Introduction of 2D CNNs to capture dimension-wise correlations.
3. **COMET (2023) ⭐**: State-of-the-art model utilizing **User History Maps** and Multi-layer CNNs.

---

### 1.4 Reference Repositories
- [NeuMF Official](https://github.com/hexiangnan/neural_collaborative_filtering)
- [ConvNCF Official](https://github.com/duxy-me/ConvNCF)
- [NeuRec Framework](https://github.com/wubinzzu/NeuRec)

---
## 2. Comparative Analysis of Model Architectures

### 2.1 Core Features by Model
| Model | Core Architecture | Key Characteristics | Relationship with COMET |
| :--- | :--- | :--- | :--- |
| **NeuMF** | GMF + MLP | Combines Linear (GMF) & Non-linear (MLP) kernels | Baseline framework for COMET |
| **ConvNCF** | Outer Product + CNN | Explicitly learns dimension-wise interactions | Advanced CNN structure used in COMET |
| **COMET** | History Map + CNN | Past Context + Dimensional Interactions | The latest evolutionary model (Proposed) |

---

### 2.2 Input Data Comparison
| Model | Input Format | Dimension | Description |
| :--- | :--- | :--- | :--- |
| **NeuMF** | User ID, Item ID | 2 IDs | Simple User-Item pairs |
| **ConvNCF** | User Emb, Item Emb | D × D Matrix | Interaction map generated via Outer Product |
| **COMET** | History Items + Target | (N+1) × D Map | User's past N items + Current target item |

---

### 2.3 Model Complexity & Efficiency
| Model | Parameters | Training Time | Inference Speed | Complexity |
| :--- | :--- | :--- | :--- | :--- |
| **NeuMF** | Medium | Fast (22s) | Fast (1.4s) | Low |
| **ConvNCF** | High | Slow (72s) | Very Slow (128s) | High |
| **COMET** | Highest | Medium (36s) | Slow (45s) | Very High |

---

### 2.4 Experimental Setup
| Parameter | Configuration Value |
| :--- | :--- |
| **Framework** | PyTorch 2.x + Cornac 2.3.5 |
| **Dataset** | Synthetic Data (Power-law distribution) |
| **Users / Items** | 1,000 / 500 |
| **Interactions** | 15,000 (Sparsity: 3%) |
| **Data Split** | Train: 70% / Val: 10% / Test: 20% |
| **Metrics** | NDCG@5/10, Recall@5/10, Precision@5/10 |
| **Epochs** | 5 |
| **Batch Size** | 256 |
| **Learning Rate** | 0.001 |
| **Embedding Dim** | 32 |
| **Optimizer** | Adam |
| **Loss Function** | BPR (Bayesian Personalized Ranking) |

## 3. Environment Setup & Data Preparation

### 3.1 Install Required Libraries
Install the necessary packages for deep learning and recommendation systems.

In [ ]:
%pip install torch numpy pandas matplotlib tqdm cornac -q

### 3.2 Library Imports
Import essential libraries and customized recommendation models.

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
import warnings
import random

warnings.filterwarnings('ignore')

# PyTorch
import torch

# Cornac (Recommendation Framework)
import cornac
from cornac.data import Reader
from cornac.eval_methods import RatioSplit
from cornac.metrics import NDCG, Recall, Precision

# Model Imports
from cornac.models import NeuMF
from convncf_cornac import ConvNCF
from comet_cornac import COMET

### 3.3 Seeding for Reproducibility
Fixing random seeds to ensure consistent experimental results.

In [ ]:
def set_seed(seed=77):
    """
    Fix all random seeds for reproducibility.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Execute Seeding
set_seed(77)

# Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"🚀 Device: {device}")
print(f"✅ Cornac version: {cornac.__version__}")
print("✅ Models imported successfully!")

### 3.4 Library Overview

| Library | Role | Use Case |
| :--- | :--- | :--- |
| **PyTorch** | Deep Learning Framework | Neural layers, Backpropagation, Optimization |
| **Cornac** | Recommendation Framework | Data splitting, Evaluation, Experiment management |
| **NumPy** | Numerical Computation | Array processing, Statistical calculations |
| **Pandas** | Data Analysis | Table creation, Statistical summaries |
| **Matplotlib** | Visualization | Performance graphs, Bar charts |

#### Key Features of Cornac Framework
* **Specialized for RecSys:** Provides built-in datasets like MovieLens and Amazon.
* **Automated Evaluation:** Simplifies Train/Val/Test splitting and metric computation.
* **Model Comparison:** Enables simultaneous training and benchmarking of multiple models.
* **Extensibility:** Easy to integrate custom models like **ConvNCF** and **COMET**.

## 4. Synthetic Data Generation

### 4.1 Data Generation Function
We generate synthetic recommendation data following a **Power-law distribution** to simulate real-world user-item interaction patterns.

In [ ]:
def generate_synthetic_data(num_users=1000, num_items=500, 
                            num_interactions=15000, seed=42):
    """
    Generates synthetic recommendation data in Cornac format.
    
    Args:
        num_users: Number of users
        num_items: Number of items
        num_interactions: Total number of interactions to generate
        seed: Random seed for reproducibility
        
    Returns:
        data: List of tuples [(user_id, item_id, rating), ...]
    
    Characteristics:
        - Item Popularity: Power-law (Zipf) distribution (long-tail effect)
        - User Activity: Gamma distribution (some users are more active)
        - Implicit Feedback: Binary ratings (1.0)
    """
    np.random.seed(seed)

    print(f"\n📊 Generating Synthetic Data...")
    print(f"- Users: {num_users}")
    print(f"- Items: {num_items}")
    print(f"- Target Interactions: {num_interactions}")

    # 1. Generate item popularity using Zipf distribution
    item_popularity = np.random.zipf(1.5, num_items)
    item_popularity = item_popularity / item_popularity.sum()

    # 2. Generate user activity levels using Gamma distribution
    user_activity = np.random.gamma(2, 2, num_users)
    user_activity = user_activity / user_activity.sum()

    # 3. Interaction Generation (Avoiding duplicates)
    data = []
    seen = set()

    while len(data) < num_interactions:
        # Select user based on activity level
        user = np.random.choice(num_users, p=user_activity)
        
        # 80% Popular items, 20% Random items (Simulating real patterns)
        if np.random.rand() < 0.8:
            item = np.random.choice(num_items, p=item_popularity)
        else:
            item = np.random.randint(0, num_items)

        if (user, item) not in seen:
            data.append((f"user_{user}", f"item_{item}", 1.0))
            seen.add((user, item))

    print(f"✅ Successfully generated {len(data)} interactions!")
    return data

### 4.2 Execute Generation
Running the function to create the dataset.

In [ ]:
synthetic_data = generate_synthetic_data(
    num_users=1000,
    num_items=500,
    num_interactions=15000,
    seed=42
)

print('\n✅ Data Generation Complete!')